# Checking the device

In [1]:
import torch

print("Is a ROCm-GPU detected? ", torch.cuda.is_available())
print("How many ROCm-GPUs are detected? ", torch.cuda.device_count())

Is a ROCm-GPU detected?  True
How many ROCm-GPUs are detected?  1


/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


# Library setup

In [2]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
import json
import os

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/redis/connection.py:77: UserWarning: redis-py works best with hiredis. Please consider installing
  warnings.warn(msg)


# Dataset preparation

In [3]:
os.getcwd()

'/workspace/opendrive_generation'

In [4]:
# Load dataset from JSONL
dataset = load_dataset(
    "json",
    data_files="/workspace/opendrive_generation/xodr_generated_scenarios_20250624_161625/scenario_metadata.jsonl",
    split="train",
)

# Shuffle for training variety
dataset = dataset.shuffle(seed=42)


# Combine prompt and response into a training text field
def format_example(example):
    prompt = example.get("prompt", "").strip()

    if "script_path" not in example:
        raise RuntimeError("No script_path in the json entry.")

    script_path = example.get("script_path", "")
    code = ""

    with open(f"{os.getcwd()}/{script_path}", "r") as code_file:
        code = code_file.read().strip()

    response = example.get("response", "").strip()
    return {
        "text": f"### Prompt:\n{prompt}\n\n### Response:\n{code}",
        "prompt": prompt,
        "response": code,
    }


# Apply mapping
dataset = dataset.map(format_example)

# Example check
print(dataset[0]["text"])

### Prompt:
Start with a straight road and gently curve it away using a spiral shape.

### Response:
from scenariogeneration import xodr, prettyprint, ScenarioGenerator

class Scenario(ScenarioGenerator):
    def __init__(self):
        super().__init__()

    def road(self):
        road = xodr.create_road(
            [xodr.Line(30), xodr.Spiral(-0.00001, -0.035, 200)],
            id=2,
            left_lanes=2,
            right_lanes=2
        )
        odr = xodr.OpenDrive("line_spiral_combo")
        odr.add_road(road)
        odr.adjust_roads_and_lanes()
        return odr

if __name__ == "__main__":
    sce = Scenario()
    prettyprint(sce.road().get_element())
    sce.generate(".")


# Model

In [5]:
# Load base model to GPU memory.
device = "cuda:0"

# Base + Adapter paths
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or Mistral etc.
ADAPTER_DIR = "llm_xodr_finetuned_model"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    trust_remote_code=True
)

# Apply LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

model.eval()

/opt/conda/envs/py_3.12/lib/python3.12/site-packages/torch/cuda/__init__.py:736: UserWarning: Can't initialize amdsmi - Error code: 34
  warnings.warn(f"Can't initialize amdsmi - Error code: {e.err_code}")


g++ (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [6]:
lengths = []

for example in dataset:
    text = f"### Prompt:\n{example['prompt']}\n\n### Response:\n{example['response']}"
    tokens = tokenizer(text)["input_ids"]
    lengths.append(len(tokens))

import numpy as np
print(f"Mean: {np.mean(lengths):.1f}, Median: {np.median(lengths)}, 95th percentile: {np.percentile(lengths, 95)}")

Mean: 386.5, Median: 319.0, 95th percentile: 701.0


In [7]:
def tokenize(batch):
    prompts = [f"### Prompt:\n{p}\n\n### Response:\n" for p in batch["prompt"]]
    responses = batch["response"]
    full_texts = [prompt + response for prompt, response in zip(prompts, responses)]

    encodings = tokenizer(
        full_texts, padding="max_length", truncation=True, max_length=512
    )

    # Mask the prompt section in the labels
    labels = []
    for i in range(len(prompts)):
        label = encodings["input_ids"][i].copy()
        prompt_len = len(tokenizer(prompts[i])["input_ids"])
        label[:prompt_len] = [-100] * prompt_len
        labels.append(label)

    encodings["labels"] = labels
    return encodings


tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# Inference testing

In [ ]:
prompt = "Make a road that curves gently to the right. Do not add any comments, I want only the code."
inputs = tokenizer(f"### Prompt:\n{prompt}\n\n### Response:\n", return_tensors="pt").to(
    "cuda"
)
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    temperature=0.0
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Prompt:
Make a road that curves gently to the right. Do not add any comments, I want only the code.

### Response:
<|assistant|>
Here's a road code that curves gently to the right using only straight lines and no comments:

```python
from math import pi, sin, cos

class Road:
    def __init__(self):
        self.road_length = 100
        self.lanes = 2
        self.left_lanes = []
        self.right_lanes = []

        left_lanes_length = int(self.road_length / (self.lanes - 1))
        right_lanes_length = int(self.road_length / (self.lanes + 1))

        left_lanes_left_lanes = []
        left_lanes_right_lanes = []
        for I in range(self.lanes - 1):
            left_lanes_left_lanes.append(Lane(i * left_lanes_length, 0, "left"))
            left_lanes_right_lanes.append(Lane(i * left_lanes_length + left_lanes_length / 2, 0, "right"))

        right_lanes_left_lanes = []
        right_lanes_right_lanes = []
        for I in range(self.lanes + 1):
            right_lanes_left